In [2]:
pip install tensorflow tensorflow-hub

  Using cached tensorflow-2.21.0-cp313-cp313-win_amd64.whl.metadata (4.5 kB)
  Using cached tensorflow_hub-0.16.1-py2.py3-none-any.whl.metadata (1.3 kB)
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.5/351.2 MB 1.5 MB/s eta 0:04:02
   ---------------------------------------- 1.0/351.2 MB 1.9 MB/s eta 0:03:08
   ---------------------------------------- 1.6/351.2 MB 2.0 MB/s eta 0:02:56
   ---------------------------------------- 2.1/351.2 MB 2.1 MB/s eta 0:02:49
   ---------------------------------------- 2.4/351.2 MB 2.1 MB/s eta 0:02:49
   ---------------------------------------- 2.9/351.2 MB 2.1 MB/s eta 0:02:44
   ---------------------------------------- 3.4/351.2 MB 2.2 MB/s eta 0:02:41
   ---------------------------------------- 3.9/351.2 MB 2.2 MB/s eta 0:02:40
    --------------------------------------- 4.5/351.2 MB 2.2 MB/s eta 0:02:40
   


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import os
import random

import numpy as np
import pandas as pd

import tensorflow as tf
import tensorflow_hub as hub

SEED = 123

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# =========================================================
# STEP 1 — Load data and integrity checks
# Model 02: ELMo + Logistic Regression
# =========================================================

CSV_PATH = "../Dati/Processed/dataset_processed_quantile1_sentences.csv"

KEEP_COLS = [
    "article_id",
    "topic_id",
    "binary_label",
    "fold",
    "text_bert"
]

# Load dataset
df = pd.read_csv(CSV_PATH)
df = df[KEEP_COLS].copy()
df = df.reset_index(drop=True)

# -----------------------------
# Integrity checks
# -----------------------------

print(f"[shape] {df.shape[0]} docs, {df.shape[1]} cols")

print("\n[missing values]")
print(df.isna().sum())

print(f"\n[duplicate article_id] {df['article_id'].duplicated().sum()}")

print(f"\n[folds] {sorted(df['fold'].unique())}")

# -----------------------------
# Label distribution
# -----------------------------

print("\n[label balance overall]")
print(
    df["binary_label"]
      .value_counts(normalize=True)
      .round(3)
      .to_dict()
)

print("\n[label balance per fold]")
print(
    df.groupby("fold")["binary_label"]
      .value_counts(normalize=True)
      .round(2)
      .unstack()
)

print("\n[documents per fold]")
print(
    df["fold"]
      .value_counts()
      .sort_index()
      .to_dict()
)

# -----------------------------
# Text length statistics
# -----------------------------

text_len = df["text_bert"].str.split().str.len()

print("\n[text length (words)]")
print(text_len.describe().round(1))

[shape] 624 docs, 5 cols

[missing values]
article_id      0
topic_id        0
binary_label    0
fold            0
text_bert       0
dtype: int64

[duplicate article_id] 0

[folds] [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

[label balance overall]
{1: 0.667, 0: 0.333}

[label balance per fold]
binary_label     0     1
fold                    
0             0.33  0.67
1             0.33  0.67
2             0.33  0.67
3             0.33  0.67
4             0.33  0.67

[documents per fold]
{0: 126, 1: 126, 2: 126, 3: 123, 4: 123}

[text length (words)]
count     624.0
mean      184.4
std       134.2
min        28.0
25%       100.8
50%       160.0
75%       239.5
max      1278.0
Name: text_bert, dtype: float64


In [11]:
import tensorflow as tf
import tensorflow_hub as hub

# =========================================================
# STEP 2 — Load ELMo model
# Model 02: ELMo + Logistic Regression
# =========================================================

print("Loading ELMo model...")

elmo = hub.load("https://tfhub.dev/google/elmo/3")

print("ELMo loaded successfully!")

Loading ELMo model...
ELMo loaded successfully!


In [12]:
# =========================================================
# STEP 3 — ELMo embeddings
# Model 02: ELMo + Logistic Regression
# =========================================================

def elmo_embeddings(texts, batch_size=32):

    embeddings = []

    texts = texts.astype(str).tolist()

    for i in range(0, len(texts), batch_size):

        batch = texts[i:i + batch_size]

        emb = elmo.signatures["default"](
            tf.constant(batch)
        )["default"]

        embeddings.append(emb.numpy())

        print(
            f"\rProcessed {min(i + batch_size, len(texts))}/{len(texts)}",
            end=""
        )

    print()

    return np.vstack(embeddings)

In [ ]:
# =========================================================
# STEP 4 — Compute ELMo embeddings
# Model 02: ELMo + Logistic Regression
# =========================================================

print("=" * 60)
print("Computing ELMo embeddings...")
print("=" * 60)

X_all = elmo_embeddings(df["text_bert"])

print("\nEmbedding matrix shape:")
print(X_all.shape)

Computing ELMo embeddings...
Processed 32/624

In [ ]:
# =========================================================
# STEP 5 — Logistic Regression + Grid Search
# Model 02: ELMo + Logistic Regression
# =========================================================

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, ParameterGrid

# ---------------------------------------------------------
# Logistic Regression
# ---------------------------------------------------------

clf = LogisticRegression(
    class_weight="balanced",
    max_iter=3000,
    random_state=SEED
)

# ---------------------------------------------------------
# Hyperparameter grid
# ---------------------------------------------------------

param_grid = {

    "C": [
        0.01,
        0.1,
        1,
        10
    ],

    "solver": [
        "liblinear"
    ],

    "penalty": [
        "l2"
    ]
}

# ---------------------------------------------------------
# Inner Cross Validation
# ---------------------------------------------------------

inner_cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=SEED
)

# ---------------------------------------------------------
# Grid Search
# ---------------------------------------------------------

grid = GridSearchCV(
    estimator=clf,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=inner_cv,
    n_jobs=-1,
    refit=True,
    verbose=1
)

print(f"Number of parameter combinations: {len(ParameterGrid(param_grid))}")

In [ ]:
# =========================================================
# STEP 6 — 5-fold CV with Grid Search
# Model 02: ELMo + Logistic Regression
# =========================================================

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report
)

oof_rows = []
fold_scores = []

N_FOLDS = 5

for fold in range(N_FOLDS):

    print("=" * 60)
    print(f"FOLD {fold}")
    print("=" * 60)

    # -------------------------------------------------
    # Train / Test split
    # -------------------------------------------------

    train_mask = df["fold"] != fold
    test_mask = df["fold"] == fold

    X_train = X_all[train_mask.values]
    X_test = X_all[test_mask.values]

    y_train = df.loc[train_mask, "binary_label"]
    y_test = df.loc[test_mask, "binary_label"]

    article_ids = df.loc[test_mask, "article_id"]

    # -------------------------------------------------
    # Grid Search
    # -------------------------------------------------

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_

    print("\nBest parameters:")
    print(grid.best_params_)

    print(f"Best inner CV Macro F1: {grid.best_score_:.4f}")

    # -------------------------------------------------
    # Prediction
    # -------------------------------------------------

    y_pred = best_model.predict(X_test)

    y_prob = best_model.predict_proba(X_test)[:, 1]

    # -------------------------------------------------
    # Metrics
    # -------------------------------------------------

    fold_f1 = f1_score(
        y_test,
        y_pred,
        average="macro"
    )

    fold_acc = accuracy_score(
        y_test,
        y_pred
    )

    fold_scores.append(fold_f1)

    print(f"\nMacro F1 : {fold_f1:.4f}")
    print(f"Accuracy : {fold_acc:.4f}")

    print("\nClassification report")
    print(
        classification_report(
            y_test,
            y_pred,
            digits=3
        )
    )

    # -------------------------------------------------
    # Save OOF predictions
    # -------------------------------------------------

    for aid, yt, yp, pr in zip(
        article_ids,
        y_test,
        y_pred,
        y_prob
    ):

        oof_rows.append({

            "article_id": int(aid),
            "fold": fold,
            "y_true": int(yt),
            "y_pred": int(yp),
            "prob_class1": float(pr)

        })

In [ ]:
# =========================================================
# STEP 7 — Overall CV results
# Model 02: ELMo + Logistic Regression
# =========================================================

oof_df = pd.DataFrame(oof_rows)

print("\n" + "=" * 60)
print("CROSS-VALIDATION RESULTS")
print("=" * 60)

print(f"Macro F1 per fold : {[round(s, 4) for s in fold_scores]}")

print(
    f"Macro F1 mean ± std : "
    f"{np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}"
)

print(f"\nOOF predictions: {len(oof_df)}")
print(f"Dataset size    : {len(df)}")

assert len(oof_df) == len(df), "Missing OOF predictions!"

# ---------------------------------------------------------
# Overall metrics from OOF predictions
# ---------------------------------------------------------

print("\nOverall OOF performance")

print(
    classification_report(
        oof_df["y_true"],
        oof_df["y_pred"],
        digits=3
    )
)

overall_f1 = f1_score(
    oof_df["y_true"],
    oof_df["y_pred"],
    average="macro"
)

overall_acc = accuracy_score(
    oof_df["y_true"],
    oof_df["y_pred"]
)

print(f"Overall Macro F1 : {overall_f1:.4f}")
print(f"Overall Accuracy : {overall_acc:.4f}")

In [ ]:
# =========================================================
# STEP 8 — Confusion Matrix
# Model 02: ELMo + Logistic Regression
# =========================================================

from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))

ConfusionMatrixDisplay.from_predictions(
    oof_df["y_true"],
    oof_df["y_pred"],
    display_labels=["Class 0", "Class 1"],
    cmap="Blues",
    values_format="d",
    ax=ax
)

ax.set_title("Confusion Matrix (OOF Predictions)")

plt.tight_layout()
plt.show()